### Ingest pit_stops.json file

#### Step 1 - Read the JSON file using spark dataframe reader API

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

In [0]:
pit_stops_schema = StructType(fields=[
    StructField("raceId", IntegerType(), False),
    StructField("driverId", IntegerType(), True),
    StructField("stop", StringType(), True),
    StructField("lap", IntegerType(), True),
    StructField("time", StringType(), True),
    StructField("duration", StringType(), True),
    StructField("milliseconds", IntegerType(), True)
])

In [0]:
pit_stops_df = spark.read \
.schema(pit_stops_schema) \
.option("multiLine", True) \
.json("abfss://raw@formula1dl2026c.dfs.core.windows.net/pit_stops.json")

#### Step 2 - Rename columns and add new columns
1. Rename driverId and raceId
2. Add ingestion_date with current timestamp

In [0]:
from pyspark.sql.functions import current_timestamp

In [0]:
final_df = pit_stops_df \
    .withColumnRenamed("raceId", "race_id") \
    .withColumnRenamed("driverId", "driver_id") \
    .withColumn("ingestion_date", current_timestamp())


#### Step 3 - Write to output to processed container in parquet format

In [0]:
final_df.write.mode("overwrite").parquet("abfss://processed@formula1dl2026c.dfs.core.windows.net/pit_stops")